# VisionTrack AI - 01: Person Detection Evaluation
This notebook evaluates the YOLO lightweight pedestrian detection module.
Evaluates Precision, Recall, mAP@50, and device inference FPS.


In [ ]:
import sys, os, time
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt

# Add backend to path
sys.path.append(str(Path.cwd().parent / "backend"))
from app.ml.detector import PersonDetector
from app.config import settings

print(f"Active Compute Device: {settings.device}")
detector = PersonDetector()


In [ ]:
# Benchmark Inference Speed (FPS)
sample_frame = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)

# Warmup
for _ in range(5):
    _ = detector.detect(sample_frame)

times = []
for _ in range(30):
    start = time.perf_counter()
    _ = detector.detect(sample_frame)
    times.append(time.perf_counter() - start)

avg_time = np.mean(times)
fps = 1.0 / avg_time
print(f"Average Inference Latency: {avg_time*1000:.2f} ms")
print(f"Throughput: {fps:.2f} FPS on {settings.device}")


In [ ]:
# Visual Detection Verification
test_img = np.full((480, 640, 3), 40, dtype=np.uint8)
cv2.circle(test_img, (320, 160), 25, (200, 180, 160), -1)
cv2.rectangle(test_img, (290, 190), (350, 320), (30, 30, 30), -1)
cv2.rectangle(test_img, (300, 320), (340, 420), (120, 70, 30), -1)

detections = detector.detect(test_img)
print(f"Detections returned: {len(detections)}")
for det in detections:
    print(f"Class: {det['class_name']}, Conf: {det['confidence']:.4f}, BBox: {det['bbox']}")
